In [0]:
from delta.tables import DeltaTable


# Func to create,insert or upsert
def upsert_to_gold(df, target_schema, target_table, join_key):
    full_table_path = f"{target_schema}.{target_table}"
    if not spark.catalog.tableExists(full_table_path):
        print(f"🚀 Table {full_table_path} does not exist. Creating new Delta table...")
        df.write.format("delta") \
          .mode("overwrite") \
          .option("overwriteSchema", "true") \
          .saveAsTable(full_table_path)
    else:
        print(f"🔄 Table {full_table_path} exists. Performing Delta Merge (Upsert)...")
        target_delta_table = DeltaTable.forName(spark, full_table_path)
        (target_delta_table.alias("target")
            .merge(
                df.alias("source"),
                f"target.{join_key} = source.{join_key}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())

        print(f"✅ Upsert successful for {target_table}")